# Лабораторна 5. Донавчання моделей (Fine-Tuning)

У цій лабораторній роботі ви ознайомитеся з різними підходами і способами донавчання моделей: класичним донавчанням з новою головою, шаблонами розмови й SFT з LoRA і QLoRA, а потім донавчите модель під власну задачу.

Усі демо-матеріали лекції 6 зібрано в архів **`06-fine-tuning-demo.zip`** — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. В архіві три демо-ноутбуки: `01_bert_head.ipynb`, `02_chat_templates.ipynb` і `03_sft_lora.ipynb`.

## Матеріали

- [Hugging Face LLM Course — розділ 3: Fine-tuning a pretrained model](https://huggingface.co/learn/llm-course/chapter3/1) — класичне донавчання BERT з новою головою.
- [Hugging Face LLM Course — розділ 11: Fine-tune Large Language Models](https://huggingface.co/learn/llm-course/chapter11/1) — шаблони розмови, SFT, PEFT, LoRA, TRL.
- [Fine-tuning vs Continued Pretraining](https://medium.com/@heyamit10/fine-tuning-vs-continued-pretraining-c8058e5040cf) — чим відрізняються донавчання й continued pretraining і коли що обирати.
- [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes) — як працюють QLoRA і формат NF4.

---
#### **Devil's Advocate UA — Dataset Generation Prompt v1.0**

Ти — генератор високоякісного синтетичного датасету для донавчання мовних моделей, які виконують роль критичного мислителя («Адвокат диявола»). Твоя задача — створювати реалістичні людські виправдання, що містять певну логічну пастку, та формувати до них конкретну, аргументовану критику.

МЕТА ДАТАСЕТУ
Датасет призначений для навчання мовної моделі аналізувати аргументацію людини, виявляти помилки мислення, самосаботаж, когнітивні упередження та нераціональні виправдання, а також формувати конструктивну контраргументацію.

Модель повинна навчитися:
- визначати слабке місце аргументації;
- пояснювати, чому наведене виправдання може бути нераціональним;
- демонструвати можливі негативні наслідки дотримання такої логіки;
- пропонувати альтернативний погляд або раціональніший спосіб дії.

Критика повинна бути логічною, конкретною, конструктивною та прив'язаною до контексту.

КАТЕГОРІЇ ЛОГІЧНИХ ПАСТОК
Датасет містить рівно 5 категорій, по 200 унікальних прикладів у кожній.

1. sunk_cost — помилка незворотних витрат
Людина продовжує вкладати час, гроші, зусилля або ресурси в рішення, проєкт чи діяльність лише тому, що вже багато вклала, ігноруючи майбутню доцільність.

2. procrastination — прокрастинація
Людина відкладає важливу дію, завдання або рішення, переконуючи себе, що пізніше, завтра або в інший момент буде кращий час, хоча це відкладання не має достатнього обґрунтування.

3. false_simplicity — хибна простота
Людина недооцінює складність, ризики, обсяг або важливість завдання, вважаючи, що воно настільки просте, що не потребує підготовки, перевірки, уваги чи планування.

4. overconfidence — надмірна самовпевненість
Людина переоцінює власні знання, здібності, досвід або точність суджень, відкидає перевірку, зворотний зв'язок, допомогу чи альтернативні думки без достатніх підстав.

5. other — інші форми самосаботажу
Інші логічні пастки, не охоплені попередніми чотирма категоріями.

Можливі механізми:
- страх невдачі;
- перфекціонізм;
- негативне прогнозування;
- уникнення відповідальності;
- порівняння себе з іншими;
- мислення за принципом «все або нічого»;
- страх осуду;
- необґрунтовані переконання про власні обмеження;
- відмова від можливостей через страх змін;
- самообмежувальні переконання.

Не використовуй у категорії other приклади, які фактично належать до sunk_cost, procrastination, false_simplicity або overconfidence, навіть якщо їх сформульовано іншими словами.

КІЛЬКІСТЬ ТА БАЛАНС
Згенеруй рівно 1 000 унікальних прикладів.

Розподіл:
- sunk_cost — 200;
- procrastination — 200;
- false_simplicity — 200;
- overconfidence — 200;
- other — 200.

Кожна категорія повинна бути представлена рівномірно.

Генеруй дані послідовно в такому порядку:
1. sunk_cost — 200;
2. procrastination — 200;
3. false_simplicity — 200;
4. overconfidence — 200;
5. other — 200.

ФОРМАТ ОДНОГО ЗАПИСУ
Кожен запис повинен бути окремим JSON-об'єктом на одному рядку у форматі JSONL.

Структура:

{"id":"sc_0001","trap_type":"sunk_cost","thesis":"...","critique":"1) Дірка: ...\n2) Якщо так: ...\n3) Але: ..."}

Поля:
- id — унікальний ідентифікатор запису;
- trap_type — назва категорії;
- thesis — теза-виправдання людини;
- critique — критика тези у форматі трьох пунктів.

Формат ID:
- sunk_cost: sc_0001 — sc_0200;
- procrastination: pr_0001 — pr_0200;
- false_simplicity: fs_0001 — fs_0200;
- overconfidence: oc_0001 — oc_0200;
- other: ot_0001 — ot_0200.

ВИМОГИ ДО ТЕЗ
Кожна теза повинна:
- бути українською мовою;
- містити від 10 до 25 слів;
- складатися рівно з одного речення;
- звучати природно, як реальне людське виправдання;
- бути конкретною та правдоподібною;
- чітко відповідати зазначеній категорії;
- не повторювати інші тези ані дослівно, ані за змістом.

Не створюй штучних, абсурдних або надто очевидних виправдань. Тези мають бути достатньо переконливими, щоб людина справді могла так думати або говорити.

РІЗНОМАНІТНІСТЬ КОНТЕКСТІВ
Використовуй широкий спектр життєвих ситуацій. Не концентруйся лише на навчанні.

Розподіляй приклади між різними контекстами, зокрема:
- навчання, університет, курсові, лабораторні, іспити;
- програмування, IT, технічні завдання, розробка;
- робота, кар'єра, співбесіди, професійні рішення;
- фінанси, покупки, бюджет, економічні рішення;
- особисті проєкти, творчість, хобі;
- повсякденні справи, побут, планування;
- стосунки, комунікація, соціальні ситуації;
- спорт, саморозвиток, звички;
- командна робота та організаційні рішення;
- підприємництво та бізнес.

Уникай надмірного повторення одного контексту. Кожна категорія повинна охоплювати багато різних сфер.

ВИМОГИ ДО КРИТИКИ
Критика повинна містити рівно три рядки, без додаткових абзаців.

Кожен рядок має виконувати окрему функцію.

1) Дірка:
Виявляє конкретну логічну помилку, необґрунтоване припущення, упередження, слабке місце або суперечність у тезі.

Не використовуй абстрактну критику. Пояснюй, що саме не так із логікою людини.

2) Якщо так:
Показує конкретний можливий наслідок, якщо людина продовжить діяти відповідно до такого виправдання.

Наслідок повинен логічно випливати з тези. Не перебільшуй і не вигадуй катастрофічних наслідків без підстав.

3) Але:
Пропонує альтернативний погляд, контраргумент, раціональніше рішення або конструктивний спосіб дії.

Альтернатива повинна бути практичною та відповідати конкретній ситуації.

СТИЛЬ КРИТИКИ
- Будь прямим, але не агресивним.
- Критикуй аргумент, а не особистість.
- Не принижуй людину.
- Не використовуй сарказм, образи або моральне засудження.
- Не став психологічних діагнозів.
- Не використовуй загальні фрази, які підходять до будь-якої тези.
- Не повторюй однакові конструкції критики в різних прикладах.
- Уникай надмірно формальної, роботизованої або академічно перевантаженої мови.
- Пиши природною сучасною українською.
- Кожна критика повинна бути змістовною навіть без прочитання інших записів датасету.

ЗАБОРОНА ШАБЛОННОСТІ ТА ДУБЛЮВАННЯ
Кожен приклад повинен відрізнятися:
- ситуацією;
- формулюванням виправдання;
- логічною слабкістю;
- можливим наслідком;
- запропонованою альтернативою.

Не використовуй просту заміну імен, професій, предметів або чисел у вже створеному прикладі для формування нового запису.

Не допускай повторення тез або близьких за змістом сценаріїв.

Не використовуй однакові аргументи в трьох пунктах критики для різних записів.

ТЕХНІЧНІ ВИМОГИ
- Виводь лише JSONL-записи.
- Один JSON-об'єкт на рядок.
- Кожен рядок повинен бути валідним JSON.
- Не додавай Markdown-оформлення, кодові блоки або пояснення.
- Не додавай коми після останнього елемента JSON.
- Використовуй лише чотири поля: id, trap_type, thesis, critique.
- У critique використовуй символи \n для розділення трьох пунктів.
- Переконайся, що thesis містить від 10 до 25 слів.
- Переконайся, що critique містить рівно три пункти з відповідними префіксами.
- Переконайся, що trap_type відповідає логіці тези.
- Переконайся, що id не повторюється.

КОНТРОЛЬ ЯКОСТІ ПЕРЕД ВИВЕДЕННЯМ
Перед виведенням кожної партії перевір:
1. Чи відповідає теза заданій категорії?
2. Чи звучить вона природно та правдоподібно?
3. Чи не повторюється вона з попередніми прикладами?
4. Чи є критика конкретною та логічною?
5. Чи виконує кожен із трьох пунктів свою функцію?
6. Чи не є критика шаблонною?
7. Чи дотримано формату JSON?
8. Чи відповідає кількість записів заданому діапазону?
9. Чи немає змішування категорій?
10. Чи є приклад корисним для навчання моделі-критика?

Якщо запис не відповідає вимогам, виправ його до виведення.

РЕЖИМ ГЕНЕРАЦІЇ
Генеруй дані партіями по 20 записів.

Не намагайся вивести всі 1 000 записів в одному повідомленні.

Після кожної партії з 20 записів зупиняйся та очікуй команди користувача «далі».

Не переходь до наступної партії самостійно.

Дотримуйся порядку категорій і точного лічильника записів.

Починай з sunk_cost.

Завершуй кожну партію повідомленням:
=== ПРОДОВЖУЙ ===

Не додавай жодних пояснень, коментарів або тексту поза JSONL-записами та маркером завершення партії.

---
## 1. Власне донавчання

Придумайте прикладну задачу, для якої промпту недостатньо, або промпт матиме свої недоліки, і розвʼяжіть її донавчанням: оберіть модель і дані, виміряйте, як модель працює до донавчання, донавчіть її й порівняйте результат. Вид донавчання — будь-який з розглянутих у лекції 6: класичне донавчання з новою головою, SFT з LoRA чи QLoRA, DPO тощо. Тренування має вміститися в безкоштовний Google Colab з GPU T4 або у ваше власне залізо.

За основу коду можна взяти демо-ноутбуки `01_bert_head` і `03_sft_lora`, але задача, дані й аналіз мають бути вашою роботою.

---
### 1.1. Задача

Опишіть задачу й те, як її результат використовуватиметься на практиці: хто користувач, що подається на вхід і що має бути на виході. Поясніть, чому для неї потрібне саме донавчання, а не промпт чи RAG.

Приклади ідей (можна обрати власну):
- класифікація відгуків чи звернень українською мовою за власним переліком категорій;
- витягування полів з оголошень, резюме чи рецептів у JSON заданої схеми;
- перетворення запитів природною мовою на SQL-запити до конкретної бази даних;
- відповіді в заданому стилі чи форматі, наприклад короткі пояснення термінів для школярів;
- виявлення спаму або токсичних коментарів.

Метою проєкту є створення моделі-критика, яка аналізує аргументи та самовиправдання, виявляє логічні помилки й когнітивні пастки та формує конструктивну критику. Основна сфера застосування - ситуації, пов'язані зі студентським життям, навчанням, виконанням завдань, програмуванням, дедлайнами та командною роботою.

Користувач подає на вхід текстову тезу, у якій міститься певне виправдання або аргумент. Модель повинна визначити тип логічної пастки, до якої належить теза, та сформувати відповідь у структурованому форматі. Критика має складатися з трьох частин: визначення логічної проблеми, пояснення можливих наслідків такого мислення та пропозиції конструктивної альтернативи.

Для класифікації використовуються п'ять категорій: sunk_cost, procrastination, false_simplicity, overconfidence та other.

Вибір задачі зумовлений результатами попереднього експерименту з локальним запуском Qwen3-1.7B, у якому модель не продемонструвала достатньої якості критичного аналізу. Зокрема, зміна квантизації не дала помітного покращення змісту відповідей, а prompt engineering переважно покращив дотримання формату, проте не забезпечив стабільного логічного аналізу та узагальнення. Тому було вирішено дослідити можливість адаптації моделі до поставленої задачі за допомогою supervised fine-tuning із застосуванням LoRA.

---
### 1.2. Модель і дані

- **Модель.** Оберіть модель під задачу й обґрунтуйте вибір: base чи instruct, розмір.
- **Дані.** Візьміть готовий датасет (Hugging Face, Kaggle тощо) або зберіть власний. Вкажіть джерело, ліцензію, кількість прикладів і формат. Покажіть кілька прикладів.
- **Поділ.** Розділіть дані на тренувальний, валідаційний і тестовий набори. Тестові приклади не повинні потрапляти в тренування.


Для донавчання обрано Qwen3-1.7B (Qwen/Qwen3-1.7B). Модель має 1,7 млрд параметрів і є компактним варіантом, що дозволяє проводити експерименти з донавчанням у середовищі Google Colab на GPU T4 без необхідності використання значних обчислювальних ресурсів. Вибір також пов'язаний із попереднім експериментом із локальним запуском Qwen3-1.7B, під час якого було виявлено недостатню якість логічного аналізу для поставленої задачі. Це стало підставою дослідити, чи можна покращити її поведінку за допомогою спеціалізованого донавчання.Текст, выделенный полужирным шрифтом

Завантажуємо датасет з Hugging Face

[посилання](https://huggingface.co/datasets/Doshik143/student-devils-advocate) **[CC BY 4.0]**

In [3]:
!pip install -q "datasets" "transformers>=4.55.0" "trl>=1.17" "peft" "accelerate"

from datasets import load_dataset
import json

dataset = load_dataset("Doshik143/student-devils-advocate", split="train")
print(f"Завантажено {len(dataset)} прикладів")
print(f"Колонки: {dataset.column_names}")
print(f"\nПерший приклад:")
print(dataset[0])

README.md:   0%|          | 0.00/6.00k [00:00<?, ?B/s]

student_critic_dataset.jsonl:   0%|          | 0.00/1.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Завантажено 1000 прикладів
Колонки: ['id', 'trap_type', 'thesis', 'critique']

Перший приклад:
{'id': 'sc_0001', 'trap_type': 'sunk_cost', 'thesis': 'Я вже написав половину курсового на WinForms, тому переходити на WPF зараз було б марною тратою часу.', 'critique': '1) Дірка: Написана частина коду не доводить, що WinForms залишається найкращим вибором для вимог курсової.\n2) Якщо так: Можеш витратити решту часу на реалізацію, яка ускладнить підтримку або не відповідатиме потрібному функціоналу.\n3) Але: Порівняй витрати переходу з перевагами WPF, враховуючи обсяг уже готового коду та вимоги проєкту.'}


Перетворюємо у формат messages

In [4]:
SYSTEM_PROMPT = "Ти — критик. Визнач тип логічної пастки в тезі й дай коротку критику у форматі JSON."

def to_messages(example):
    """Перетворює один приклад у формат messages для SFTTrainer."""
    answer = json.dumps({
        "trap_type": example["trap_type"],
        "critique": example["critique"],
    }, ensure_ascii=False)
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["thesis"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

dataset = dataset.map(to_messages, remove_columns=dataset.column_names)
print(f"Перетворено {len(dataset)} прикладів")
print(f"\nПерший приклад у форматі messages:")
print(json.dumps(dataset[0]["messages"], ensure_ascii=False, indent=2))

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Перетворено 1000 прикладів

Перший приклад у форматі messages:
[
  {
    "content": "Ти — критик. Визнач тип логічної пастки в тезі й дай коротку критику у форматі JSON.",
    "role": "system"
  },
  {
    "content": "Я вже написав половину курсового на WinForms, тому переходити на WPF зараз було б марною тратою часу.",
    "role": "user"
  },
  {
    "content": "{\"trap_type\": \"sunk_cost\", \"critique\": \"1) Дірка: Написана частина коду не доводить, що WinForms залишається найкращим вибором для вимог курсової.\\n2) Якщо так: Можеш витратити решту часу на реалізацію, яка ускладнить підтримку або не відповідатиме потрібному функціоналу.\\n3) Але: Порівняй витрати переходу з перевагами WPF, враховуючи обсяг уже готового коду та вимоги проєкту.\"}",
    "role": "assistant"
  }
]


Розділяємо на train/val/test (80/10/10)

In [5]:
shuffled = dataset.shuffle(seed=42)
split_1 = shuffled.train_test_split(test_size=0.2, seed=42)
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split_1["train"]
val_dataset = split_2["train"]
test_dataset = split_2["test"]

print(f"Train: {len(train_dataset)} ({len(train_dataset)/len(dataset)*100:.0f}%)")
print(f"Val:   {len(val_dataset)} ({len(val_dataset)/len(dataset)*100:.0f}%)")
print(f"Test:  {len(test_dataset)} ({len(test_dataset)/len(dataset)*100:.0f}%)")

Train: 800 (80%)
Val:   100 (10%)
Test:  100 (10%)


---
### 1.3. Модель до донавчання

Оберіть метрику, яка відповідає задачі (accuracy, F1, частка валідних відповідей, точність окремих полів тощо), і поясніть вибір. Оцініть модель на тестовому наборі до донавчання:
- для LLM — з найкращим промптом, який вдасться скласти;
- для класифікатора з новою головою — порівняйте з найпростішим базовим рівнем, наприклад з відповіддю «завжди найчастіший клас».

Наведіть 5–10 прикладів відповідей моделі.

Для оцінювання використовувалися дві метрики:

- Accuracy (trap_type) - частка прикладів, для яких модель правильно визначила категорію логічної пастки.

- Valid JSON - частка відповідей, які відповідають вимогам JSON-формату та можуть бути коректно оброблені програмно.

Завантажуємо модель й функцію для генерації

In [1]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-1.7B"

print(f"Завантаження {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, device_map="auto"
)
print(f"Модель завантажена. Пам'ять GPU: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


def classify(model, thesis, system_prompt, max_new_tokens=400):
    """Генерує відповідь моделі на одну тезу (без режиму міркування)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": thesis},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def parse_answer(text):
    """Витягує JSON з відповіді моделі (може бути в ```json ... ```)."""
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start >= 0 and end > start:
            data = json.loads(text[start:end])
            if "trap_type" in data:
                return data
    except json.JSONDecodeError:
        pass
    return None


def evaluate(model, dataset, system_prompt):
    """Рахує accuracy визначення trap_type на датасеті."""
    correct = 0
    valid = 0
    examples = []
    for ex in dataset:
        thesis = ex["messages"][1]["content"]
        target = json.loads(ex["messages"][2]["content"])["trap_type"]
        answer = classify(model, thesis, system_prompt)
        parsed = parse_answer(answer)
        if parsed:
            valid += 1
            if parsed["trap_type"] == target:
                correct += 1
        examples.append((thesis, answer, target))
    n = len(dataset)
    return {
        "accuracy": correct / n * 100,
        "valid_json": valid / n * 100,
    }, examples

Завантаження Qwen/Qwen3-1.7B...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Модель завантажена. Пам'ять GPU: 3.4 GB


In [ ]:
LONG_PROMPT = """Ти — критик. Визнач тип логічної пастки в тезі й дай коротку критику у форматі JSON.

Перелік типів:
- sunk_cost: витрачені ресурси
- procrastination: відкладання
- false_simplicity: хибна простота
- overconfidence: надмірна впевненість
- other: інше

Формат відповіді: {"trap_type": "<тип>", "critique": "..."}"""

print("Оцінка «до» донавчання (baseline)...")
metrics_before, examples_before = evaluate(model, test_dataset, LONG_PROMPT)

print(f"\n=== Результати baseline ===")
print(f"Accuracy (trap_type): {metrics_before['accuracy']:.1f}%")
print(f"Valid JSON: {metrics_before['valid_json']:.1f}%")

print(f"\n=== 5 прикладів відповідей моделі ===")
for i, (thesis, answer, target) in enumerate(examples_before[:5], 1):
    print(f"\n{i}. Теза: {thesis[:80]}...")
    print(f"   Очікуваний тип: {target}")
    print(f"   Відповідь моделі: {answer[:150]}...")

Оцінка «до» донавчання (baseline)...

=== Результати baseline ===
Accuracy (trap_type): 14.0%
Valid JSON: 34.0%

=== 5 прикладів відповідей моделі ===

1. Теза: Я не буду розбирати механізм Dependency Injection, бо достатньо створювати потрі...
   Очікуваний тип: false_simplicity
   Відповідь моделі: {"trap_type": "procrastination", "critique": "Ти використовуєш відкладання (procrastination) через відмову в розгляданні механізму Dependency Injectio...

2. Теза: Я не буду розбирати адаптивний дизайн, бо достатньо встановити ширину контейнера...
   Очікуваний тип: false_simplicity
   Відповідь моделі: {"trap_type": "false_simplicity", "critique": "Теза відкидає необхідність розгляду адаптивного дизайну через встановлення ширини контейнера у відсотка...

3. Теза: Я не буду розбирати принципи роботи планувальника ОС, бо достатньо запустити пот...
   Очікуваний тип: false_simplicity
   Відповідь моделі: {"trap_type": "procrastination", "critique": "Теза відкладає розуміння основних принципі

In [6]:
SHORT_PROMPT = "Ти — критик. Визнач тип логічної пастки в тезі й дай коротку критику у форматі JSON."

print("Оцінка «до» донавчання (baseline)...")
metrics_before, examples_before = evaluate(model, test_dataset, SHORT_PROMPT)

print(f"\n=== Результати baseline ===")
print(f"Accuracy (trap_type): {metrics_before['accuracy']:.1f}%")
print(f"Valid JSON: {metrics_before['valid_json']:.1f}%")

print(f"\n=== 5 прикладів відповідей моделі ===")
for i, (thesis, answer, target) in enumerate(examples_before[:5], 1):
    print(f"\n{i}. Теза: {thesis[:80]}...")
    print(f"   Очікуваний тип: {target}")
    print(f"   Відповідь моделі: {answer[:150]}...")

Оцінка «до» донавчання (baseline)...

=== Результати baseline ===
Accuracy (trap_type): 0.0%
Valid JSON: 0.0%

=== 5 прикладів відповідей моделі ===

1. Теза: Я не буду розбирати механізм Dependency Injection, бо достатньо створювати потрі...
   Очікуваний тип: false_simplicity
   Відповідь моделі: Ось приклад типу логічної пастки (logical packaging) в тезі, що відповідає вашому ставленню:

```json
{
  "type": "logical-packaging",
  "description"...

2. Теза: Я не буду розбирати адаптивний дизайн, бо достатньо встановити ширину контейнера...
   Очікуваний тип: false_simplicity
   Відповідь моделі: {
  "type": "пастка",
  "teza": "Я не буду розбирати адаптивний дизайн, бо достатньо встановити ширину контейнера у відсотках, щоб сайт автоматично пі...

3. Теза: Я не буду розбирати принципи роботи планувальника ОС, бо достатньо запустити пот...
   Очікуваний тип: false_simplicity
   Відповідь моделі: Ось приклад JSON-форматованої критики у вигляді логічної пастки:

```json
{
  "type": "cri

---
### 1.4. Донавчання

Донавчіть модель. Поясніть вибір методу й основних гіперпараметрів: learning rate, кількість кроків чи епох, розмір батча, для LoRA — `r` і `lora_alpha`. Покажіть таблицю чи графік training і validation loss, вкажіть тривалість тренування й пікову памʼять GPU. Якщо помітите перенавчання, поясніть, що змінили.

Для адаптації базової моделі Qwen3-1.7B до задачі визначення логічних пасток у студентських аргументах було використано метод Supervised Fine-Tuning (SFT) із застосуванням LoRA (Low-Rank Adaptation).

LoRA дозволяє адаптувати модель без повного перенавчання всіх її параметрів. Замість зміни основних ваг моделі навчання відбувається через додаткові адаптери, що зменшує кількість параметрів, які необхідно оптимізувати, та потреби у відеопам'яті.

У процесі навчання використовувалися 800 записів навчальної вибірки. Валідаційна вибірка з 100 записів застосовувалася для контролю процесу навчання, а тестова вибірка не використовувалася під час навчання.

Для навчання було обрано такі параметри:

- LoRA rank (r): 16

- LoRA alpha: 32

- LoRA dropout: 0.05

- Learning rate: 0.0002

- Scheduler: cosine

- Warmup steps: 5

- Max training steps: 200

- Effective batch size: 8

- Max sequence length: 512

- Precision: FP16

- Gradient checkpointing: enabled

Навчання виконувалося з використанням GPU T4 у середовищі Google Colab. Такий підхід дозволив адаптувати модель до специфічного формату відповідей і навчити її визначати задані категорії логічних пасток та формувати структуровану критику.

Налаштування LoRA

In [ ]:
!pip uninstall -q -y torchao

In [ ]:
!pip install -q "trl>=1.17" "peft"

from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
import time

# --- LoRA конфігурація ---
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

# --- Параметри тренування ---
training_args = SFTConfig(
    output_dir="qwen3-1.7b-devils-advocate-lora",
    max_length=512,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    max_steps=200,
    eval_strategy="steps",
    eval_steps=20,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    assistant_only_loss=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

# --- Тренер ---
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.model.print_trainable_parameters()

Tokenizing train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Запуск тренування

In [ ]:
import torch

torch.cuda.reset_peak_memory_stats()
start_time = time.time()

trainer.train()

train_time = (time.time() - start_time) / 60
peak_memory = torch.cuda.max_memory_allocated() / 1e9

print(f"\n✅ Тренування завершено!")
print(f"Тривалість: {train_time:.1f} хв")
print(f"Пікова пам'ять GPU: {peak_memory:.1f} GB")

# Зберігаємо адаптер
trainer.save_model("qwen3-1.7b-devils-advocate-lora-adapter")
print("Адаптер збережено.")

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
20,1.218981,1.074798,0.999064,0.744039,55664.000000
40,0.974993,0.960118,0.929369,0.762105,111972.000000
60,0.920001,0.915168,0.911033,0.770689,167880.000000
80,0.910978,0.889469,0.888900,0.776925,223731.000000
100,0.908128,0.872532,0.890744,0.778221,279790.000000
120,0.763816,0.865547,0.767639,0.781180,335947.000000
140,0.765602,0.858561,0.772382,0.783427,391760.000000
160,0.765120,0.854300,0.758278,0.785304,447921.000000
180,0.732012,0.851548,0.758228,0.786412,503301.000000
200,0.734488,0.851087,0.757212,0.786001,559580.000000



✅ Тренування завершено!
Тривалість: 13.1 хв
Пікова пам'ять GPU: 4.3 GB
Адаптер збережено.


Оцінка «після» донавчання

In [ ]:
SHORT_PROMPT = "Ти — критик. Визнач тип логічної пастки в тезі й дай коротку критику у форматі JSON."

print("Оцінка «після» донавчання...")
metrics_after, examples_after = evaluate(trainer.model, test_dataset, SHORT_PROMPT)

print(f"\n=== Результати після донавчання ===")
print(f"Accuracy (trap_type): {metrics_after['accuracy']:.1f}%")
print(f"Valid JSON: {metrics_after['valid_json']:.1f}%")

print(f"\n=== Порівняння ===")
print(f"{'Метрика':<25} {'До':>10} {'Після':>10}")
print("-" * 47)
print(f"{'Accuracy (trap_type)':<25} {metrics_before['accuracy']:>9.1f}% {metrics_after['accuracy']:>9.1f}%")
print(f"{'Valid JSON':<25} {metrics_before['valid_json']:>9.1f}% {metrics_after['valid_json']:>9.1f}%")

print(f"\n=== 5 прикладів після донавчання ===")
for i, (thesis, answer, target) in enumerate(examples_after[:5], 1):
    print(f"\n{i}. Теза: {thesis[:80]}...")
    print(f"   Очікуваний тип: {target}")
    print(f"   Відповідь моделі: {answer[:150]}...")

Оцінка «після» донавчання...

=== Результати після донавчання ===
Accuracy (trap_type): 97.0%
Valid JSON: 100.0%

=== Порівняння ===
Метрика                           До      Після
-----------------------------------------------
Accuracy (trap_type)           14.0%      97.0%
Valid JSON                     34.0%     100.0%

=== 5 прикладів після донавчання ===

1. Теза: Я не буду розбирати механізм Dependency Injection, бо достатньо створювати потрі...
   Очікуваний тип: false_simplicity
   Відповідь моделі: {"trap_type": "false_simplicity", "critique": "1) Дірка: використання new не пояснює, які залежності потрібні, які зміни відповідатимуть на зміну конт...

2. Теза: Я не буду розбирати адаптивний дизайн, бо достатньо встановити ширину контейнера...
   Очікуваний тип: false_simplicity
   Відповідь моделі: {"trap_type": "false_simplicity", "critique": "1) Дірка: встановлення відсотків не пояснює розміщення елементів, зміну шрифтів, збереження контенту, п...

3. Теза: Я не буду розбира

---
### 1.5. Порівняння

На тому самому тестовому наборі й тих самих прикладах порівняйте модель до і після донавчання. Проаналізуйте помилки донавченої моделі: які приклади вона розвʼязує неправильно і чому. Для LLM перевірте також на кількох загальних запитах поза вашою задачею, чи не погіршилися відповіді моделі.

Перевірка на загальних запитах (чи не зіпсувалася модель)

In [ ]:
general_questions = [
    "Яка столиця Франції?",
    "Поясни, що таке фотосинтез, у двох реченнях.",
    "Скільки буде 15 × 24?",
    "Напиши коротке привітання для друга.",
]

# Промпт без специфіки нашої задачі
GENERAL_SYSTEM = "Ти — корисний асистент. Відповідай коротко й по суті."

print("=== Перевірка на загальних запитах ===")
print("(порівнюємо донавчену модель із базовою)\n")

# Донавчена модель
print("🤖 Донавчена модель (з LoRA):")
for q in general_questions:
    answer = classify(trainer.model, q, GENERAL_SYSTEM, max_new_tokens=100)
    print(f"  ❓ {q}")
    print(f"  💬 {answer[:150]}...\n")

# Базова модель
print("=" * 60)
print("\n🤖 Базова модель (без LoRA):")
from transformers import AutoModelForCausalLM
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-1.7B", dtype=torch.float16, device_map="auto"
)
for q in general_questions:
    answer = classify(base_model, q, GENERAL_SYSTEM, max_new_tokens=100)
    print(f"  ❓ {q}")
    print(f"  💬 {answer[:150]}...\n")

del base_model
torch.cuda.empty_cache()

=== Перевірка на загальних запитах ===
(порівнюємо донавчену модель із базовою)

🤖 Донавчена модель (з LoRA):
  ❓ Яка столиця Франції?
  💬 Столиця Франції — місто Париж....

  ❓ Поясни, що таке фотосинтез, у двох реченнях.
  💬 Фотосинтез — процес, за допомогою якого рослини та бактерії відтворюють зелений кисень і створюють їжу і енергію за допомогою сонячного світла. Це дже...

  ❓ Скільки буде 15 × 24?
  💬 15 × 24 = 360....

  ❓ Напиши коротке привітання для друга.
  💬 Привіт! Якщо ти хочеш, можу допомогти з ще одним питанням або відповісти на конкретну тему. Але ти хочеш лише привітання? 😊...


🤖 Базова модель (без LoRA):


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  ❓ Яка столиця Франції?
  💬 Столиця Франції — це Париж....

  ❓ Поясни, що таке фотосинтез, у двох реченнях.
  💬 Фотосинтез — процес, за допомогою якого рослини та алгебри конвертуєть сонну енергію в хімічну. Цей процес відбувається в хлоропластах і дозволяє росл...

  ❓ Скільки буде 15 × 24?
  💬 360....

  ❓ Напиши коротке привітання для друга.
  💬 Здоров'я!...



---
### 1.6. Мій звіт:

- **Задача і чому потрібне донавчання:** Створення моделі, яка визначає логічні пастки у студентських аргументах та формує конструктивну критику. Донавчання потрібне для покращення точності класифікації та дотримання заданого формату відповідей.

- **Модель і дані:** Qwen3-1.7B та синтетичний датасет українською мовою з 1000 записів, що охоплює 5 категорій логічних пасток.

- **Метрика й результат до донавчання:** Зі складним промтом: Accuracy - 14%, Valid JSON - 34%. Базова модель іноді галюцинувала і не розуміла задачі. З простим промтом: Accuracy - 0%, Valid JSON - 0%. Базова модель не відповідала заданим вимогам класифікації та формату.

- **Метод донавчання й гіперпараметри:** Supervised Fine-Tuning із LoRA (r=16, alpha=32, dropout=0.05, learning rate=0.0002, 200 кроків навчання).

- **Результат після донавчання:** Accuracy - 97%, Valid JSON - 100%, що свідчить про значне покращення виконання поставленої задачі.

- **Аналіз помилок:** Модель правильно класифікувала 97 із 100 тестових прикладів. Водночас синтетичні дані та обмежений розмір тестової вибірки можуть впливати на узагальнення результатів.

- **Висновок:** LoRA-донавчання дозволило ефективно адаптувати Qwen3-1.7B до задачі аналізу студентських аргументів, суттєво покращивши точність класифікації та форматування відповідей.

---
### Як здавати лабу

У [форму](https://forms.gle/rJRRREjXqDGVeuLg7) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбуки з усіма виконаними пунктами.** Відповіді до 0.1 додайте новими секціями до відповідних демо-ноутбуків: питання 1–10 — до `01_bert_head.ipynb`, 11–21 — до `02_chat_templates.ipynb`, 22–30 — до `03_sft_lora.ipynb`.
2. **Ноутбук власного донавчання** `ft_project.ipynb` з виконаними клітинками й звітом до 1.6 окремою секцією.
3. Ваги моделі чи адаптера в репозиторій **не додавайте**. За бажанням опублікуйте адаптер на Hugging Face Hub і додайте посилання у звіт (за це буде +1 бонусний бал).
4. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab5/01_bert_head.ipynb
/lab5/02_chat_templates.ipynb
/lab5/03_sft_lora.ipynb
/lab5/ft_project.ipynb
```

In [ ]:
from huggingface_hub import login, HfApi

login(token="hf_")

HF_USERNAME = "Doshik143"
REPO_NAME = f"{HF_USERNAME}/qwen3-1.7b-devils-advocate-lora"

api = HfApi()
api.create_repo(repo_id=REPO_NAME, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="qwen3-1.7b-devils-advocate-lora-adapter", repo_id=REPO_NAME, repo_type="model")
print(f"✅ Адаптер назавжди тут: https://huggingface.co/{REPO_NAME}")

✅ Адаптер назавжди тут: https://huggingface.co/Doshik143/qwen3-1.7b-devils-advocate-lora


[посилання](https://huggingface.co/Doshik143/qwen3-1.7b-devils-advocate-lora)